# RecGuard + Gemini: governing MMM insights and budget recommendations

**Scenario.** AcmeHealth, the Indian online pharmacy from the previous notebook, spends about ₹9 crore a quarter on marketing across ten channels. The analytics team fits a **marketing mix model (MMM)**, a regression that estimates how much revenue each channel drives. A **Gemini** analyst agent then reads the results and writes the quarterly readout: insights, plus a budget reallocation plan for the CMO and CFO.

LLM-written MMM readouts fail in predictable ways:

| Failure | Example |
|---|---|
| **Misquoted numbers** | "YouTube ROI is 6×" when the model says 4.6× |
| **Ignoring uncertainty** | Scaling a channel whose ROI interval includes zero |
| **Confounded attribution** | Search *looks* like the best channel because it follows TV, which drives the searches |
| **Average vs marginal ROI** | Adding money to a channel with high average ROI that is already saturated |
| **Extrapolation** | Planning spend far outside anything the data has seen |
| **Governance blind spots** | A channel on brand-safety hold, a contract floor, customer data without DPDP consent, an expired contract |

RecGuard treats each insight and each budget action as a candidate *recommendation*. It checks them against the **MMM itself** and against the company's **governance registry**, signs the approved plan, and audits everything.

**Simulated data with a hidden ground truth.** No real client data is used. The weekly data is generated from known "true" channel effects that neither the MMM nor Gemini sees. That lets the notebook do what's impossible with real data: measure whether the raw LLM plan and the guarded plan actually make money.

**Cost:** two Gemini calls.

## 1. Setup

In [ ]:
%pip install -q "google-genai>=1.0" numpy -e ..

### API key

Read from `GEMINI_API_KEY` in the environment, then from the repository's gitignored `.env`, then from a hidden prompt. It's never printed.

In [ ]:
import getpass, os, pathlib

def load_key_from_dotenv(path=pathlib.Path("..") / ".env"):
    if not path.exists():
        return None
    for line in path.read_text().splitlines():
        name, sep, value = line.partition("=")
        if sep and name.strip() == "GEMINI_API_KEY":
            return value.strip().strip("'\"") or None
    return None

if not os.environ.get("GEMINI_API_KEY"):
    key = load_key_from_dotenv() or getpass.getpass("GEMINI_API_KEY (hidden): ").strip()
    if key:
        os.environ["GEMINI_API_KEY"] = key
if not os.environ.get("GEMINI_API_KEY"):
    raise ValueError("Set GEMINI_API_KEY in the environment or .env, or enter it at the prompt.")
print("GEMINI_API_KEY is set")

In [ ]:
import json, math, re, time
import numpy as np
from google import genai
from google.genai import types
from pydantic import BaseModel
from IPython.display import Markdown, display

from recguard import Candidate, Denied, Outcome, RecGuard, Request, Telemetry, load_pack

MODEL = "gemini-3.8-flash"
PRICE_PER_MTOK = {"input": 0.30, "output": 2.50, "cache_read": 0.03}  # Gemini 3.8 Flash, USD
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def table(rows, headers):
    out = "| " + " | ".join(headers) + " |\n|" + "---|" * len(headers) + "\n"
    out += "\n".join("| " + " | ".join(str(c) for c in row) + " |" for row in rows)
    display(Markdown(out))

## 2. Three years of weekly marketing data *(simulated)*

All amounts are ₹ lakh (1 lakh = 100,000). The generator builds in the traps a real MMM faces:

* **TV** runs in bursts, with a long carryover effect.
* **Search** spend follows TV (TV creates searches), so the two are almost perfectly collinear. Its *true* incremental effect is small.
* **Meta** is heavily saturated; **YouTube** and **retail media** have room to grow.
* **Influencer** ran only a handful of campaigns. **CTV** launched 10 weeks ago. **Print**'s contract ended 30 weeks ago.

`TRUTH` is used only to generate the data and, at the end, to score plans. The MMM and Gemini never see it.

In [ ]:
rng = np.random.default_rng(2026)
W = 156
t = np.arange(W)
diwali = np.zeros(W)
for start in (42, 94, 146):
    diwali[start:start + 3] = 1

TRUTH = {  # carryover decay, half-saturation (adstocked ₹ lakh/week), max weekly effect (₹ lakh revenue)
    "tv": dict(decay=0.6, half_sat=90, beta=260),          "youtube": dict(decay=0.4, half_sat=45, beta=120),
    "meta": dict(decay=0.2, half_sat=12, beta=80),         "search": dict(decay=0.1, half_sat=40, beta=14),
    "retail_media": dict(decay=0.1, half_sat=30, beta=95), "influencer": dict(decay=0.3, half_sat=25, beta=40),
    "programmatic": dict(decay=0.2, half_sat=30, beta=8),  "crm_whatsapp": dict(decay=0.1, half_sat=6, beta=30),
    "ctv": dict(decay=0.5, half_sat=20, beta=30),          "print": dict(decay=0.3, half_sat=20, beta=6),
}

SPEND = {}
tv_on = (rng.random(W) < 0.45).astype(float); tv_on[diwali > 0] = 1
SPEND["tv"] = tv_on * rng.normal(70, 12, W).clip(20)
SPEND["youtube"] = rng.normal(16, 4, W).clip(4)
SPEND["meta"] = rng.normal(26, 5, W).clip(8)
SPEND["search"] = (6 + 0.3 * SPEND["tv"] + rng.normal(0, 1.0, W)).clip(3)
SPEND["retail_media"] = rng.normal(12, 4, W).clip(2) * (1 + 0.6 * diwali)
SPEND["influencer"] = np.where(rng.random(W) < 0.08, rng.normal(30, 8, W).clip(10), 0.0)
SPEND["programmatic"] = rng.normal(11, 3, W).clip(2)
SPEND["crm_whatsapp"] = rng.normal(4, 2, W).clip(0.5)
SPEND["ctv"] = np.where(t >= W - 10, rng.normal(14, 3, W).clip(5), 0.0)
SPEND["print"] = np.where(t < W - 30, rng.normal(8, 2, W).clip(2), 0.0)
CHANNELS = list(SPEND)

def adstock(x, decay):
    out, carry = np.zeros_like(x, dtype=float), 0.0
    for i, v in enumerate(x):
        carry = v + decay * carry
        out[i] = carry
    return out

def hill(x, half_sat):
    return x / (x + half_sat)

def true_effect(c, spend):
    p = TRUTH[c]
    return p["beta"] * hill(adstock(spend, p["decay"]), p["half_sat"])

baseline = 520 + 0.6 * t + 45 * np.sin(2 * np.pi * t / 52) + 25 * np.cos(2 * np.pi * t / 52) + 160 * diwali
REVENUE = baseline + sum(true_effect(c, SPEND[c]) for c in CHANNELS) + rng.normal(0, 22, W)
print(f"{W} weeks, {len(CHANNELS)} channels, revenue ₹{REVENUE.sum()/100:,.0f} crore, media spend ₹{sum(s.sum() for s in SPEND.values())/100:,.0f} crore")

## 3. Fit the MMM

A transparent, numpy-only MMM:
* **Geometric adstock** (carryover) and **Hill saturation** for each channel, with parameters chosen by grid search and coordinate descent.
* **Non-negative regression** with trend, seasonality and a Diwali control.
* A **13-week holdout** check.
* A **block bootstrap** (300 resamples) for 90% intervals.
* **Collinearity diagnostics**: VIF, the variance inflation factor, and the highest spend correlation with any other channel.

It runs in a few seconds. A production team would use Meridian, Robyn or PyMC-Marketing. RecGuard doesn't care which; it consumes the results table.

In [ ]:
RUN_ID = "mmm-2026q3-r1"
controls = np.column_stack([np.ones(W), t / W, np.sin(2*np.pi*t/52), np.cos(2*np.pi*t/52),
                            np.sin(4*np.pi*t/52), np.cos(4*np.pi*t/52), diwali])
K = controls.shape[1]
GRID = [(d, m) for d in (0.0, 0.2, 0.4, 0.6, 0.8) for m in (0.5, 1.0, 2.0, 4.0, 8.0)]

def ref_level(c, decay):
    a = adstock(SPEND[c], decay)
    return a[a > 0].mean() if (a > 0).any() else 1.0

def feature(c, decay, mult, spend=None):
    return hill(adstock(SPEND[c] if spend is None else spend, decay), mult * ref_level(c, decay))

def nnls(X, y, rows):
    active = list(range(X.shape[1]))
    for _ in range(X.shape[1]):
        coef, *_ = np.linalg.lstsq(X[rows][:, active], y[rows], rcond=None)
        negative = [a for a, b in zip(active, coef) if a >= K and b < 0]
        if not negative:
            break
        active = [a for a in active if a not in negative]
    full = np.zeros(X.shape[1]); full[active] = coef
    return full

def design(params):
    return np.column_stack([controls] + [feature(c, *params[c]) for c in CHANNELS])

def fit(rows):
    params = {c: (0.4, 1.0) for c in CHANNELS}
    def sse(p):
        X = design(p); coef = nnls(X, REVENUE, rows)
        return ((REVENUE[rows] - X[rows] @ coef) ** 2).sum()
    for _ in range(3):
        for c in CHANNELS:
            params[c] = min(GRID, key=lambda g: sse({**params, c: g}))
    X = design(params)
    return params, X, nnls(X, REVENUE, rows)

train, holdout = np.arange(W - 13), np.arange(W - 13, W)
_, X_tr, coef_tr = fit(train)
holdout_mape = float(np.mean(np.abs(X_tr[holdout] @ coef_tr - REVENUE[holdout]) / REVENUE[holdout]))

PARAMS, X, COEF = fit(np.arange(W))
r2 = float(1 - ((REVENUE - X @ COEF) ** 2).sum() / ((REVENUE - REVENUE.mean()) ** 2).sum())

boot_rng = np.random.default_rng(7)
BOOT = []
for _ in range(300):
    starts = boot_rng.integers(0, W - 8, W // 8 + 1)
    BOOT.append(nnls(X, REVENUE, np.concatenate([np.arange(s, s + 8) for s in starts])[:W]))
BOOT = np.array(BOOT)

def model_effect(c, beta, spend):
    return beta * feature(c, *PARAMS[c], spend=spend)

def vif(j):
    others = [i for i in range(X.shape[1]) if i != j]
    b, *_ = np.linalg.lstsq(X[:, others], X[:, j], rcond=None)
    resid = X[:, j] - X[:, others] @ b
    return float(1 / max(1e-6, (resid ** 2).sum() / ((X[:, j] - X[:, j].mean()) ** 2).sum()))

MMM = {}
total_media = sum(SPEND[c].sum() for c in CHANNELS)
for i, c in enumerate(CHANNELS):
    j, spend, s = K + i, SPEND[c].sum(), SPEND[c]
    roi_draws = np.array([model_effect(c, b[j], s).sum() / spend for b in BOOT])
    mroi_draws = np.array([(model_effect(c, b[j], s * 1.01).sum() - model_effect(c, b[j], s).sum()) / (0.01 * spend) for b in BOOT])
    corr = {o: float(np.corrcoef(s, SPEND[o])[0, 1]) for o in CHANNELS if o != c and SPEND[o].std() > 0}
    partner = max(corr, key=lambda o: abs(corr[o]))
    decay = PARAMS[c][0]
    contribution = model_effect(c, COEF[j], s).sum()
    MMM[c] = {
        "weeks_active": int((s > 0).sum()), "spend_share": round(spend / total_media, 3),
        "contribution_share": round(contribution / REVENUE.sum(), 3),
        "roi": round(contribution / spend, 2),
        "roi_low": round(float(np.percentile(roi_draws, 5)), 2), "roi_high": round(float(np.percentile(roi_draws, 95)), 2),
        "mroi": round(float((model_effect(c, COEF[j], s * 1.01).sum() - contribution) / (0.01 * spend)), 2),
        "mroi_low": round(float(np.percentile(mroi_draws, 5)), 2), "mroi_high": round(float(np.percentile(mroi_draws, 95)), 2),
        "half_life_weeks": round(math.log(0.5) / math.log(decay), 1) if decay > 0 else 0.0,
        "vif": round(vif(j), 1), "max_spend_corr": round(abs(corr[partner]), 2), "corr_with": partner,
        "max_quarter_spend_observed": round(float(np.convolve(s, np.ones(13), "valid").max()), 1),
    }
print(f"run {RUN_ID}: R² {r2:.3f}, 13-week holdout MAPE {holdout_mape:.1%}")

In [ ]:
table([[c, m["weeks_active"], f'{m["spend_share"]:.0%}', f'{m["roi"]:.2f}', f'[{m["roi_low"]:.2f}, {m["roi_high"]:.2f}]',
        f'{m["mroi"]:.2f}', f'[{m["mroi_low"]:.2f}, {m["mroi_high"]:.2f}]', m["half_life_weeks"], m["vif"],
        f'{m["max_spend_corr"]:.2f} ({m["corr_with"]})'] for c, m in MMM.items()],
      ["Channel", "Weeks", "Spend share", "ROI", "ROI 90% CI", "Marginal ROI", "mROI 90% CI", "Half-life (wk)", "VIF", "Max spend corr"])

## 4. What the MMM can't know: the governance registry

These facts live in procurement, legal and brand-safety systems, not in the regression. Gemini isn't given them, just as an analyst agent reading an MMM deck wouldn't be.

The **base plan** for next quarter keeps every channel at its trailing-52-week run-rate. CTV stays at its launch run-rate, and Print is at zero because its contract has ended.

In [ ]:
REGISTRY = {c: {"available": True, "brand_safety_hold": False, "uses_customer_data": False, "contract_floor_pct": None} for c in CHANNELS}
REGISTRY["programmatic"]["brand_safety_hold"] = True     # open-exchange ads served next to unsafe content in Q3
REGISTRY["crm_whatsapp"]["uses_customer_data"] = True    # personalised WhatsApp/CRM journeys use customer data
REGISTRY["print"]["available"] = False                   # newspaper contract ended; no inventory booked
REGISTRY["tv"]["contract_floor_pct"] = -10               # annual upfront deal: at most a 10% cut this year
DPDP_CONSENT_COVERAGE = 0.38                             # share of CRM audience with valid DPDP Act consent
CONSENT_OK = DPDP_CONSENT_COVERAGE >= 0.80               # legal's threshold to scale customer-data activation

def base_quarter(c):
    if c == "ctv":
        return float(SPEND[c][-10:].mean() * 13)
    if not REGISTRY[c]["available"]:
        return 0.0
    return float(SPEND[c][-52:].sum() / 4)

BASE_PLAN = {c: round(base_quarter(c), 1) for c in CHANNELS}
QUARTER_BUDGET = round(sum(BASE_PLAN.values()), 1)
table([[c, f"₹{BASE_PLAN[c]:,.0f} L", "on hold" if r["brand_safety_hold"] else "", "yes" if r["uses_customer_data"] else "",
        "" if r["available"] else "contract ended", r["contract_floor_pct"] or ""] for c, r in REGISTRY.items()],
      ["Channel", "Base quarter spend", "Brand safety", "Customer data", "Availability", "Contract floor %"])
print(f"Quarter budget ₹{QUARTER_BUDGET:,.0f} lakh; DPDP consent coverage {DPDP_CONSENT_COVERAGE:.0%} (scaling allowed: {CONSENT_OK})")

## 5. Policies and MMM-aware checks, in YAML

Every rule lives in [`policies/mmm_insights.yaml`](policies/mmm_insights.yaml): the three policies, every threshold as a named constant, and the checks as `reject: REASON` / `if: condition` rules. The notebook only passes in the data the rules read (`MMM`, `REGISTRY`, `BASE_PLAN`). To change a threshold or add a rule, edit the YAML and re-run this cell. To check your edits, including the pack's own `tests:`, run `recguard policy validate policies/mmm_insights.yaml` from a shell.

**Global marketing governance** (insights and plans):
* Everything must cite MMM evidence.
* Customer-data activation needs consent.
* Nothing may *increase* spend on a channel under brand-safety hold.
* Decisions are `high_impact`, so the bandit never experiments with budget plans.

**Finance policy** (`budget-plan` only): at most one action per channel, and a minimum confidence of 0.4.

| Check | Rule (constant in the YAML) |
|---|---|
| `fact-check` | Every number an insight cites must match the MMM, within `fact_tolerance` (10%) or `fact_abs_tolerance` (0.05) |
| `mmm` | Increases need marginal-ROI 90% lower bound ≥ `breakeven_mroi` (1.0) |
| `mmm` | Increases on a channel whose spend correlates ≥ `confounding_corr` (0.7) with another are confounded: run a geo test |
| `mmm` | No planned quarter above `extrapolation_margin` (1.1) × the highest quarter observed |
| `mmm` | No cut where the pessimistic marginal ROI exceeds `protect_mroi_low` (1.5) |
| `finance` | At most `max_change_pct` (30%) per channel per quarter, tests ≤ `max_test_pct` (10%), contract floors |

The MMM run ID is the pack's `data_version`, so refitting the model invalidates receipts issued against the old run.

In [ ]:
guard = RecGuard(signing_key=os.urandom(32), receipt_ttl=1800)
PACK = load_pack("policies/mmm_insights.yaml")
INSIGHT_METRICS = tuple(PACK.constants["insight_metrics"])
loaded = guard.load_policy_pack(PACK, context={"MMM": MMM, "REGISTRY": REGISTRY, "BASE_PLAN": BASE_PLAN}, data_version=RUN_ID)
print(loaded)

def parse_action(item_id):  # used by the adapter and plan scoring below
    channel, action, pct = item_id.split(":")
    return channel, action, int(pct)

eff = guard.effective_policy("acmehealth", "budget-plan")
print({k: v for k, v in eff.items() if k in ("require_evidence", "denied_categories", "max_per_category", "min_score", "high_impact", "custom_checks")})

## 6. The Gemini analyst

Gemini receives the MMM table, the base plan and the brief, and returns **structured output**: an executive summary, numeric insights (channel, metric, value, statement) and budget actions (channel, action, percentage change, confidence, rationale).

In [ ]:
class Insight(BaseModel):
    channel: str
    metric: str        # one of the MMM table's metric names
    value: float       # the exact number being cited
    statement: str

class Action(BaseModel):
    channel: str
    action: str        # increase | decrease | maintain | test
    change_pct: float  # % change to the channel's quarterly spend (test: % of channel budget for a geo experiment)
    confidence: float  # 0..1
    rationale: str

class MMMReview(BaseModel):
    executive_summary: str
    insights: list[Insight]
    actions: list[Action]

LLM_MMM_VIEW = {"run_id": RUN_ID, "fit": {"r2": round(r2, 3), "holdout_mape": round(holdout_mape, 3)},
                "channels": MMM, "base_quarter_plan_lakh": BASE_PLAN, "quarter_budget_lakh": QUARTER_BUDGET}

SYSTEM = (
    "You are a senior marketing-science analyst at AcmeHealth (Indian online pharmacy). You read marketing mix model "
    "(MMM) results and write the quarterly readout for the CMO and CFO. Return: an executive summary; 5-8 insights, "
    "each citing exactly one metric value copied from the MMM table (metric must be one of: " + ", ".join(INSIGHT_METRICS)
    + "); and one budget action per channel you want to change (action: increase, decrease, maintain or test; "
    "change_pct is a positive percentage of that channel's base quarterly spend; confidence 0 to 1). Use channel names "
    "exactly as in the table.\n\nMMM RESULTS (JSON):\n" + json.dumps(LLM_MMM_VIEW)
)

def llm_review(brief):
    started = time.perf_counter()
    response = client.models.generate_content(
        model=MODEL,
        contents=brief,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM,
            max_output_tokens=16000,
            response_mime_type="application/json",
            response_schema=MMMReview,
        ),
    )
    latency_ms = (time.perf_counter() - started) * 1000
    if not isinstance(response.parsed, MMMReview):
        raise RuntimeError(f"Gemini returned no valid structured review: {response.prompt_feedback or response.candidates}")
    usage = response.usage_metadata
    input_tokens = (usage.prompt_token_count or 0) if usage else 0
    cache_read = (usage.cached_content_token_count or 0) if usage else 0
    output_tokens = ((usage.candidates_token_count or 0) + (usage.thoughts_token_count or 0)) if usage else 0
    cost = (max(0, input_tokens - cache_read) * PRICE_PER_MTOK["input"]
            + output_tokens * PRICE_PER_MTOK["output"] + cache_read * PRICE_PER_MTOK["cache_read"]) / 1e6
    meta = {"latency_ms": round(latency_ms, 1), "tokens": input_tokens + output_tokens, "cost_usd": round(cost, 6),
            "model": response.model_version or MODEL}
    return response.parsed, meta

### Adapter: LLM output → RecGuard candidates

The **evidence** and **flags** on each candidate come from the MMM and the registry, never from the LLM:
* An *increase* carries MMM evidence only if the channel has at least 26 weeks of data and an ROI interval clear of zero.
* *Decrease* and *maintain* need only that the channel exists in the MMM.
* *Tests* carry the experimentation policy as evidence.

The action's exact numbers are part of its ID, so the signed receipt binds them.

In [ ]:
SLUG = re.compile(r"[^a-z0-9_]")
ACTIONS = ("increase", "decrease", "maintain", "test")

def action_id(a):
    channel = SLUG.sub("_", a.channel.strip().lower())[:40] or "unknown"
    action = a.action.strip().lower() if a.action.strip().lower() in ACTIONS else "invalid"
    pct = 0 if action == "maintain" else int(round(abs(a.change_pct)))
    return f"{channel}:{action}:{pct}"

def to_action_candidate(a):
    item_id = action_id(a)
    channel, action, pct = parse_action(item_id)
    m, reg = MMM.get(channel), REGISTRY.get(channel)
    if m is None or action == "invalid":   # hallucinated channel or action: no evidence, rejected
        return Candidate(item_id=item_id, tenant_id="acmehealth", score=0.0)
    if action == "increase":
        evidence = [f"{RUN_ID}:{channel}:roi"] if m["weeks_active"] >= 26 and m["roi_low"] > 0 else []
    elif action == "test":
        evidence = ["experimentation-policy-2026:geo-holdout"]
    else:
        evidence = [f"{RUN_ID}:{channel}:observed"]
    expands = action in ("increase", "test")
    return Candidate(item_id=item_id, tenant_id="acmehealth", score=max(0.0, min(1.0, a.confidence)),
                     categories=[f"ch-{channel}"] + (["increase-on-brand-safety-hold"] if expands and reg["brand_safety_hold"] else []),
                     available=reg["available"] or not expands,
                     uses_personal_data=expands and reg["uses_customer_data"], evidence_ids=evidence)

def fmt(v):
    return f"{v:.4f}".rstrip("0").rstrip(".") or "0"

def to_insight_candidate(i, ins):
    channel = SLUG.sub("_", ins.channel.strip().lower())[:40] or "unknown"
    metric = SLUG.sub("_", ins.metric.strip().lower())[:30] or "unknown"
    known = channel in MMM and metric in INSIGHT_METRICS
    return Candidate(item_id=f"insight-{i}:{channel}:{metric}:{fmt(ins.value)}", tenant_id="acmehealth", score=1.0,
                     evidence_ids=[f"{RUN_ID}:{channel}:{metric}"] if known else [])

def guarded_review(brief, request_id):
    review, meta = llm_review(brief)
    common = dict(tenant_id="acmehealth", region="IN", segment="marketing-leadership", personalization_consent=CONSENT_OK)
    insights = guard.evaluate(Request(purpose="mmm-insight", request_id=f"{request_id}-insights", k=12,
                                      candidates=[to_insight_candidate(i, x) for i, x in enumerate(review.insights)], **common))
    plan = guard.evaluate(Request(purpose="budget-plan", request_id=f"{request_id}-plan", k=10,
                                  candidates=[to_action_candidate(a) for a in review.actions], **common))
    guard.event(Telemetry(event_id=f"llm-{plan.decision_id}", tenant_id="acmehealth", decision_id=plan.decision_id,
                          event_type="LATENCY", latency_ms=meta["latency_ms"], token_count=meta["tokens"], cost_usd=meta["cost_usd"]))
    return review, insights, plan, meta

def show_insights(review, decision):
    rejected = {r.item_id: r.reason for r in decision.rejections}
    rows = []
    for i, ins in enumerate(review.insights):
        cid = to_insight_candidate(i, ins).item_id
        actual = MMM.get(cid.split(":")[1], {}).get(cid.split(":")[2], "n/a")
        verdict = "✅ verified" if cid in {r.item_id for r in decision.results} else f"⛔ {rejected.get(cid, 'not served')}"
        rows.append([ins.channel, ins.metric, fmt(ins.value), actual, verdict, ins.statement])
    table(rows, ["Channel", "Metric", "Cited", "MMM value", "RecGuard", "Statement"])

def show_plan(review, decision, meta):
    rejected = {r.item_id: r.reason for r in decision.rejections}
    served = {r.item_id for r in decision.results}
    sign = {"increase": "+", "decrease": "−", "test": "test "}
    rows = [[a.channel, a.action, f"{sign.get(a.action.strip().lower(), '')}{abs(a.change_pct):.0f}%" if a.action != "maintain" else "", f"{a.confidence:.2f}",
             "✅ approved" if action_id(a) in served else f"⛔ {rejected.get(action_id(a), 'not served')}", a.rationale]
            for a in review.actions]
    display(Markdown(f"**Gemini's plan** ({meta['model']}, {meta['latency_ms']:.0f} ms, {meta['tokens']} tokens, ${meta['cost_usd']:.4f})"))
    table(rows, ["Channel", "Action", "Change", "Confidence", "RecGuard", "Rationale"])
    display(Markdown(f"**Plan status:** `{decision.status}`, {len(served)} of {len(review.actions)} actions approved.  \n**Reason codes:** {decision.reason_codes}"))

### Scoring a plan against the hidden truth

A plan becomes per-channel multipliers on next quarter's base plan. Both the fitted MMM and the hidden `TRUTH` simulate the quarter, including carryover from past spend. **Net** = incremental revenue − incremental spend.

In [ ]:
def plan_multipliers(actions):
    mult = {c: 1.0 for c in CHANNELS}
    for item_id in actions:
        channel, action, pct = parse_action(item_id)
        if channel in mult and action in ("increase", "decrease"):
            mult[channel] = 1 + (pct if action == "increase" else -pct) / 100
    return mult

def next_quarter(c, scale):
    weekly = BASE_PLAN[c] / 13 * scale
    if c == "tv":                         # keep the flighting pattern
        on = np.array([1, 1, 0, 0] * 3 + [1], dtype=float)
        return np.concatenate([SPEND[c], on * weekly * 13 / on.sum()])
    if c == "influencer":
        burst = np.zeros(13); burst[[3, 9]] = weekly * 13 / 2
        return np.concatenate([SPEND[c], burst])
    return np.concatenate([SPEND[c], np.full(13, weekly)])

def score_plan(actions):
    mult = plan_multipliers(actions)
    out = {"true_rev": 0.0, "model_rev": 0.0, "spend": 0.0}
    for i, c in enumerate(CHANNELS):
        if mult[c] == 1.0:
            continue
        new, old = next_quarter(c, mult[c]), next_quarter(c, 1.0)
        out["true_rev"] += (true_effect(c, new)[-13:] - true_effect(c, old)[-13:]).sum()
        out["model_rev"] += (model_effect(c, COEF[K + i], new)[-13:] - model_effect(c, COEF[K + i], old)[-13:]).sum()
        out["spend"] += BASE_PLAN[c] * (mult[c] - 1)
    out["true_net"] = out["true_rev"] - out["spend"]
    out["model_net"] = out["model_rev"] - out["spend"]
    return out

def compare(review, decision):
    raw = [action_id(a) for a in review.actions if parse_action(action_id(a))[0] in MMM]
    approved = [r.item_id for r in decision.results]
    rows = []
    for name, actions in [("Status quo", []), ("Gemini plan, unguarded", raw), ("RecGuard-approved plan", approved)]:
        s = score_plan(actions)
        rows.append([name, f"₹{s['spend']:+,.0f} L", f"₹{s['model_rev']:+,.0f} L", f"₹{s['model_net']:+,.0f} L",
                     f"**₹{s['true_rev']:+,.0f} L**", f"**₹{s['true_net']:+,.0f} L**"])
    table(rows, ["Plan", "Δ spend", "Δ revenue (MMM says)", "Net (MMM says)", "Δ revenue (truth)", "Net (truth)"])
    return score_plan(raw), score_plan(approved)

## 7. Brief 1: budget-neutral reallocation

> "Prepare next quarter's readout. Keep total spend flat (budget-neutral) and reallocate to maximise incremental revenue."

In [ ]:
BRIEF_1 = ("Prepare next quarter's MMM readout for the CMO and CFO. Keep total spend flat (budget-neutral) and "
           "reallocate across channels to maximise incremental revenue.")
review_1, insights_1, plan_1, meta_1 = guarded_review(BRIEF_1, "q4-realloc")
display(Markdown("**Executive summary (free text, not fact-checked):** " + review_1.executive_summary))
show_insights(review_1, insights_1)

In [ ]:
show_plan(review_1, plan_1, meta_1)
raw_1, approved_1 = compare(review_1, plan_1)

## 8. Brief 2: the CFO cuts the budget by 15%

> "Cut next quarter's total marketing spend by 15% with the smallest possible revenue loss."

Cuts are where the analyst can do the most damage, by removing spend that is actually profitable. They also run into contracts.

In [ ]:
BRIEF_2 = ("The CFO needs next quarter's total marketing spend cut by 15% versus the base plan. Recommend the cuts "
           "that lose the least incremental revenue, and say what to protect.")
review_2, insights_2, plan_2, meta_2 = guarded_review(BRIEF_2, "q4-cut15")
show_plan(review_2, plan_2, meta_2)
raw_2, approved_2 = compare(review_2, plan_2)
cut = -score_plan([r.item_id for r in plan_2.results])["spend"] / QUARTER_BUDGET
print(f"Approved cuts deliver {cut:.1%} of the 15% target; the remainder needs a human decision (RecGuard won't invent cuts).")

## 9. Sign-off: the receipt binds the exact numbers

The approved plan's receipt covers every action, including its percentage. If anyone edits "+20%" to "+45%" between approval and execution, verification fails. The CMO's sign-off is recorded as the verified presentation, and the quarter's realised result becomes the trusted outcome.

In [ ]:
plan = plan_1 if plan_1.results else plan_2
print("approved plan verifies:", guard.verify("acmehealth", plan.decision_id, plan.receipt, plan.results))
edited = [plan.results[0].model_copy(update={"item_id": re.sub(r":\d+$", ":45", plan.results[0].item_id)})] + list(plan.results[1:])
print(f"edited {plan.results[0].item_id} -> {edited[0].item_id}; verifies:", guard.verify("acmehealth", plan.decision_id, plan.receipt, edited))

try:
    guard.presented("acmehealth", plan.decision_id, plan.receipt, plan.results)   # CMO sign-off
    realised = score_plan([r.item_id for r in plan.results])["true_net"]           # simulated quarter result
    guard.outcome(Outcome(outcome_id=f"q4-close-{plan.decision_id[:12]}", tenant_id="acmehealth", decision_id=plan.decision_id,
                          reward=1.0 if realised > 0 else 0.0, outcome_type="BUSINESS_GOAL"))
    print(f"signed off; realised net ₹{realised:+,.0f} lakh recorded as a trusted outcome")
except Denied as exc:
    print("not recorded:", exc, "(already recorded, or the receipt expired / policy changed: re-run the brief)")

## 10. The verified readout

This is what reaches the CMO and CFO: only fact-checked insights and approved actions. The executive summary is free text that can't be verified, so it's excluded. A human can rewrite it from the verified points.

In [ ]:
lines = [f"### AcmeHealth MMM readout ({RUN_ID})", f"*Model fit: R² {r2:.2f}, holdout MAPE {holdout_mape:.1%}. "
         f"Policy fingerprint `{plan_1.policy_fingerprint[:12]}…`*", "", "**Verified insights**"]
served = {r.item_id for r in insights_1.results}
lines += [f"- {x.statement}" for i, x in enumerate(review_1.insights) if to_insight_candidate(i, x).item_id in served]
lines += ["", "**Approved budget actions (budget-neutral brief)**"]
approved = {r.item_id for r in plan_1.results}
lines += [f"- **{a.channel}**: {a.action} {abs(a.change_pct):.0f}%: {a.rationale}" for a in review_1.actions if action_id(a) in approved]
blocked = [(a, r.reason) for a in review_1.actions for r in plan_1.rejections if r.item_id == action_id(a)]
if blocked:
    lines += ["", "**Not approved (needs a human decision)**"]
    lines += [f"- {a.channel} {a.action} {abs(a.change_pct):.0f}%: `{reason}`" for a, reason in blocked]
display(Markdown("\n".join(lines)))

## 11. Audit and metrics

In [ ]:
head = guard.db.audit_head("acmehealth")
print("audit chain:", guard.db.audit_verify("acmehealth", head["head"])["valid"], "| entries:", head["entries"])
m = guard.metrics("acmehealth")
table([[k, m[k]] for k in ["decisions", "statuses", "repair_rate", "eligibility_rate", "verified_presentations", "outcomes",
                            "mean_reward", "latency_ms_p50", "token_count_logged", "cost_usd_logged"]], ["Metric", "Value"])
table(sorted(m["reason_counts"].items(), key=lambda x: -x[1]), ["Reason code", "Decisions affected"])

## What this shows

| Failure in an LLM-written MMM readout | RecGuard reason code |
|---|---|
| Misquoted or invented numbers | `fact-check:NUMBER_DOES_NOT_MATCH_MMM`, `METRIC_NOT_IN_MMM` |
| Scaling a channel with too little data or an interval spanning zero | `EVIDENCE_REQUIRED` |
| Scaling a confounded channel (Search follows TV) | `mmm:CONFOUNDED_ATTRIBUTION_RUN_GEO_TEST` |
| Adding money where marginal ROI may be below breakeven (saturation) | `mmm:INCREASE_NOT_PROFITABLE_AT_MARGIN` |
| Planning beyond observed spend | `mmm:EXTRAPOLATES_BEYOND_OBSERVED_SPEND` |
| Cutting clearly profitable spend | `mmm:CUTS_CLEARLY_PROFITABLE_SPEND` |
| Moves too big, tests too big, contract floors | `finance:*` |
| Brand-safety hold, DPDP consent, expired contract | `DENIED_CATEGORY`, `CONSENT_REQUIRED`, `UNAVAILABLE` |
| Contradictory actions for the same channel | `CATEGORY_CAP` |
| Edited numbers after approval | receipt verification fails |

**Reading the truth comparison honestly:**
* Gemini's plan differs from run to run, so compare the rows in each run's own table.
* RecGuard doesn't *optimise* a plan; it removes moves the evidence doesn't support. That usually protects the downside, as with scaling Search, which the MMM over-credits.
* It can also leave value on the table. CRM/WhatsApp is genuinely profitable in the hidden truth, but the MMM can't detect that (its ROI interval includes zero, so `EVIDENCE_REQUIRED` fires). Even with better evidence, scaling it would hit `CONSENT_REQUIRED` until DPDP consent coverage reaches legal's threshold.
* TV and Search are almost perfectly collinear, so the confounding rule blocks increases on *both*. The data can't separate their effects, which is exactly why the reason code asks for a geo test.
* Here the MMM itself overestimates YouTube and retail media. RecGuard checks *against* the MMM, so it can only be as right as the model. Geo tests, which the `CONFOUNDED_ATTRIBUTION_RUN_GEO_TEST` code points to, are how teams calibrate it.